## Import Libraries

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Import Dataset

In [0]:
df = pd.read_csv('/Workspace/Users/ajiboyeniola@gmail.com/lead-scoring/data/raw/leads.csv')
df.head()

In [0]:
df.info()

## Shape of the dataset

In [0]:
print("Number of rows and columns in the dataset")
df.shape

In [0]:
# Compare unique counts to total rows
total_rows = len(df)
for col in df.columns:
    if df[col].nunique() == total_rows:
        print(f"Column '{col}' is a unique identifier.")

In [0]:
print("Unique values for each colmn")
df.nunique()

## Missing Values

In [0]:
missing_values = (df.isnull().sum() / len(df) * 100).round(2)
missing_values = missing_values[missing_values > 0]
missing_values = missing_values.sort_values(ascending=True)
missing_values

In [0]:
plt.figure(figsize=(10, 6))
sns.barplot(
    x=missing_values.index, 
    y=missing_values.values,
    order=missing_values.index, 
    palette="viridis", 
    legend=False)
plt.title("Percentage of Missing Values per Column")
plt.xlabel("Column")
plt.ylabel("Percentage of Missing Values")
plt.show()

In [0]:
df.describe()

## Quantitative Distribution

In [0]:
plot_cols = ['email_opens', 'email_clicks', 'website_visits', 'pages_viewed', 'content_downloads', 'num_contacts', 'ad_clicks', 'days_since_last_contact', 'lead_age_days']

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(plot_cols):
    sns.histplot(df[col].dropna(), bins=10, kde=True, ax=axes[i])
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

plt.suptitle('Quantitative Features Distribution')
plt.tight_layout()
plt.show()

## Outlier Analysis

In [0]:
fig, axes = plt.subplots(3,3, figsize=(15,12))
axes = axes.flatten()

for i, col in enumerate(plot_cols):
    axes[i].boxplot(df[col].dropna())
    axes[i].set_title(f'Boxplot of {col}')
    axes[i].set_ylabel(col)

plt.suptitle('Boxplots of Quantitative Features')
plt.tight_layout()
plt.show()

In [0]:
for col in plot_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f'Number of outliers in {col}: {len(outliers)}')


## Target Distribution

In [0]:
df['converted'].value_counts()

In [0]:
sns.histplot(df['converted'], bins=2)
plt.title('Distribution of Converted')
plt.xlabel('Converted')
plt.ylabel('Count')
plt.show()

In [0]:
df.dtypes

## Categorical Distribution



In [0]:
ordinal_cols = ['company_size', 'funnel_stage']
nominal_cols = ['business_type', 'lead_source', 'ad_platform', 'preferred_contact_time']

orders = {
    'company_size': ['Small', 'Medium', 'Large'],
    'funnel_stage': ['Awareness', 'Interest', 'Consideration', 'Decision']
}

all_cols = ordinal_cols + nominal_cols

fig, axes = plt.subplots(3, 2, figsize=(15, 15))
axes = axes.flatten()

for i, col in enumerate(all_cols):
    if col in ordinal_cols:
        order = orders[col]
    else:
        order = df.groupby(col)['converted'].mean().sort_values(ascending=False).index
    sns.barplot(
        x=col, 
        y='converted', 
        data=df,
        order=order,
        ax=axes[i],
        errorbar=None
    )
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Conversion Rate')
    axes[i].tick_params(axis='x', rotation=45)


plt.tight_layout()
plt.show()

In [0]:
print(df['lead_source'].value_counts())
print(df['ad_platform'].value_counts())

## Feature vs Target

In [0]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(plot_cols):
    sns.boxplot(x='converted', y=col, data=df, ax=axes[i])
    axes[i].set_xlabel('Converted(0 = No, 1 = Yes)')
    axes[i].set_ylabel(col)
    axes[i].set_title(f'{col} vs Converted')
    # axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [0]:
pd.set_option('display.max_columns', None)
print(df.groupby('converted')[plot_cols].median())

## Correlation heatmap

In [0]:
plt.figure(figsize=(12, 8))
correlation_matrix = df[plot_cols + ['converted']].corr()

sns.heatmap(
    correlation_matrix, 
    annot=True, 
    cmap='coolwarm', 
    center=0,
    fmt='.2f',
    linewidths=0.5
)

plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()